# 04 — SHAP Explainability

SHAP (SHapley Additive exPlanations) analysis for the XGBoost churn model:
- Global feature importance
- Individual prediction explanations
- Feature interaction effects

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import warnings
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

from src.features.actuarial_features import ActuarialFeatureBuilder

warnings.filterwarnings("ignore")
shap.initjs()

DATA_PATH = Path("../data/churn_dataset.parquet")

In [ ]:
df = pd.read_parquet(DATA_PATH)
X = df.drop(columns=["churn_label", "policy_id"])
y = df["churn_label"]

builder = ActuarialFeatureBuilder()
X_features = builder.fit_transform(X)

drop_cols = ["lob", "channel"]
X_model = X_features.drop(columns=[c for c in drop_cols if c in X_features.columns])

split_idx = int(len(X_model) * 0.80)
X_train, X_test = X_model.iloc[:split_idx], X_model.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

xgb = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=3,
    random_state=42, eval_metric="auc", verbosity=0,
)
xgb.fit(X_train, y_train)
print(f"Test AUC: {__import__('sklearn.metrics', fromlist=['roc_auc_score']).roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1]):.4f}")

## 1. SHAP Summary Plot (Global Feature Importance)

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, show=True)

## 2. SHAP Bar Plot (Mean Absolute SHAP)

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type="bar", show=True)

## 3. Individual Prediction Explanations

In [ ]:
high_risk_idx = np.argsort(xgb.predict_proba(X_test)[:, 1])[-1]
print(f"Highest risk policyholder (index {high_risk_idx}):")
print(f"Churn probability: {xgb.predict_proba(X_test)[high_risk_idx, 1]:.2%}")
print(f"Actual label: {'Churned' if y_test.iloc[high_risk_idx] == 1 else 'Retained'}")

shap.force_plot(
    explainer.expected_value,
    shap_values[high_risk_idx, :],
    X_test.iloc[high_risk_idx, :],
    matplotlib=True,
)
plt.show()

In [ ]:
low_risk_idx = np.argsort(xgb.predict_proba(X_test)[:, 1])[0]
print(f"Lowest risk policyholder (index {low_risk_idx}):")
print(f"Churn probability: {xgb.predict_proba(X_test)[low_risk_idx, 1]:.2%}")

shap.force_plot(
    explainer.expected_value,
    shap_values[low_risk_idx, :],
    X_test.iloc[low_risk_idx, :],
    matplotlib=True,
)
plt.show()

## 4. SHAP Dependence Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, feature in zip(axes, ["premium_to_market_ratio", "tenure_years", "has_recent_claim"]):
    if feature in X_test.columns:
        shap.dependence_plot(
            feature, shap_values, X_test,
            ax=ax, show=False,
        )

plt.suptitle("SHAP Dependence Plots — Key Churn Drivers", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Waterfall Plot (Single Prediction)

In [ ]:
shap_explanation = shap.Explanation(
    values=shap_values[high_risk_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[high_risk_idx].values,
    feature_names=X_test.columns.tolist(),
)
shap.waterfall_plot(shap_explanation, show=True)

## Key Insights

- **`premium_to_market_ratio`** is consistently the top predictor — overpriced policies churn
- **`tenure_years`** has a protective effect — loyal customers are sticky
- **`is_multi_line`** reduces churn risk — bundling creates switching cost
- **`slow_settlement`** and **`has_unsettled_claim`** drive churn through negative experience
- **`is_young_adult`** increases churn — younger customers are more price-sensitive

These SHAP explanations satisfy regulatory explainability requirements (EU AI Act, Solvency II).